# Refresh Tracker

Rebuilds the **Summary** tab and the **CSV exports** from the Scoring grid.

## Why this exists

The Summary tab used to be built from live Excel formulas. It rendered as 0% across every row. openpyxl writes the formula string but strips the cached result, and Excel would not honour the recalculate-on-load flag we set. Two attempts to fix that in place both failed.

So the Summary tab now contains **static values** computed here in Python. They display unconditionally, in any spreadsheet application, with no dependency on Excel evaluating anything.

## The trade

Nothing updates by itself any more. If you change a verdict on the Scoring tab, the Summary and the exports are stale until you re-run this notebook. The sheet carries a **last refreshed** stamp so staleness is visible rather than silent.

## Run All, not cell by cell

> This notebook **deletes and rebuilds** the Summary sheet and **overwrites** the Scoring Score column. Use **Run All**. Running cells individually, out of order, or twice against leftover state risks leaving the workbook half-rebuilt in a way that looks fine and is not.
>
> The rebuild is deliberately kept in a single cell so it cannot be partially executed.

Close the workbook in Excel before running. The first cell will stop with a clear message if it is still open.

In [1]:
import csv
from collections import Counter
from datetime import datetime
from pathlib import Path

import openpyxl
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side


def find_project_root(marker="Papers"):
    # Walk upward until we find the folder containing Papers/, that is the project root.
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / marker).is_dir():
            return c
    raise FileNotFoundError(f"Could not locate project root (looking for a '{marker}' folder).")


ROOT  = find_project_root()
XL    = ROOT / "Data" / "Tracker" / "checklist_scoring_tracker.xlsx"
OUT   = ROOT / "Data" / "Tracker" / "exports"

FIRST_ROW, LAST_ROW = 2, 16          # scoring data rows
SCOREABLE = ("Yes", "No", "Unclear") # N/A and blanks are excluded from every denominator
STAMP = datetime.now().strftime("%Y-%m-%d %H:%M")

if (XL.parent / ("~$" + XL.name)).exists():
    raise SystemExit("Workbook is open in Excel. Close it and re-run.")

print(f"Project root : {ROOT}")
print(f"Tracker      : {XL}  (exists: {XL.exists()})")
print(f"Stamp        : {STAMP}")

Project root : C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal
Tracker      : C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Data\Tracker\checklist_scoring_tracker.xlsx  (exists: True)
Stamp        : 2026-08-08 18:05


In [2]:
# Styling and the one helper every section uses.
F     = "Arial"
HFILL = PatternFill("solid", start_color="1F3864", end_color="1F3864")
SFILL = PatternFill("solid", start_color="D9E2F3", end_color="D9E2F3")
HFONT = Font(name=F, bold=True, color="FFFFFF", size=10)
SFONT = Font(name=F, bold=True, size=10)
BODY  = Font(name=F, size=10)
BOLD  = Font(name=F, size=10, bold=True)
ITAL  = Font(name=F, size=9, italic=True)
THIN  = Side("thin", color="B7B7B7")
BR    = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
WRAP  = Alignment(wrap_text=True, vertical="top", horizontal="left")
CTR   = Alignment(horizontal="center", vertical="center")
SCALE = dict(start_type="num", start_value=0,   start_color="F4B183",
             mid_type="num",   mid_value=0.5,   mid_color="FFE699",
             end_type="num",   end_value=1,     end_color="C6E0B4")

wb    = openpyxl.load_workbook(XL, data_only=False)
sc    = wb["Scoring"]
ITEMS = [sc.cell(1, c).value for c in range(4, 17)]
ROWS  = list(range(FIRST_ROW, LAST_ROW + 1))
DOMS  = {d: [r for r in ROWS if sc.cell(r, 2).value == d]
         for d in ("Retinal", "Colonoscopy", "ECG")}


def rate(rows, col):
    # Returns (yes, scoreable, proportion) for one checklist column over the given rows.
    v = [sc.cell(r, col).value for r in rows]
    y = sum(1 for x in v if x == "Yes")
    d = sum(1 for x in v if x in SCOREABLE)
    return y, d, (y / d if d else None)


GROUPS = [
    ("MODEL-DEVELOPMENT \u2014 does the model work?  (items 1, 13)", [0, 12]),
    ("TRIAL CONDUCT \u2014 was the study run properly?  (items 3, 4, 5, 10, 11, 12)", [2, 3, 4, 9, 10, 11]),
    ("INTEGRITY \u2014 honest reporting, disclosed conflicts  (items 7, 8)", [6, 7]),
    ("CLINICAL IMPACT \u2014 does it help patients?  (items 2, 9)", [1, 8]),
    ("   of which, patient outcomes alone  (item 9)", [8]),
    ("TRANSPARENCY \u2014 is the model inspectable?  (item 6)", [5]),
]

# Scope revised 2026-08-02 after reviewer feedback, logged as deviation D7. A 12-lead ECG
# is a time series and not an image, so the PRIMARY analysis covers the imaging papers only.
# The ECG papers are retained as a prespecified SECONDARY set, reported separately and never
# pooled into a headline figure.
PRIMARY_DOMAINS = ("Retinal", "Colonoscopy")


def set_of(domain):
    return "PRIMARY" if domain in PRIMARY_DOMAINS else "SECONDARY"


PRIMARY_ROWS   = [r for r in ROWS if set_of(sc.cell(r, 2).value) == "PRIMARY"]
SECONDARY_ROWS = [r for r in ROWS if set_of(sc.cell(r, 2).value) == "SECONDARY"]
assert len(PRIMARY_ROWS) + len(SECONDARY_ROWS) == len(ROWS)

print(f"{len(ROWS)} papers, {len(ITEMS)} items, {len(GROUPS)} groups. Workbook loaded.")
print(f"  PRIMARY (imaging)   {len(PRIMARY_ROWS)} papers")
print(f"  SECONDARY (ECG)     {len(SECONDARY_ROWS)} papers")

15 papers, 13 items, 6 groups. Workbook loaded.
  PRIMARY (imaging)   11 papers
  SECONDARY (ECG)     4 papers


## Rebuild the workbook

One cell, deliberately. It does two things that must either both happen or neither:

1. Writes the **static Yes-count** into the Scoring tab's Score column, replacing the formulas that were rendering as zero.
2. Deletes and rebuilds the **Summary** tab with static values across all four sections plus the notes block.

Then it saves. If this cell is interrupted partway, re-run it from the top rather than continuing.

In [3]:
# ---- job 1: static Yes-counts in the Scoring Score column ----------------
for r in ROWS:
    y = sum(1 for c in range(4, 17) if sc.cell(r, c).value == "Yes")
    cell = sc.cell(r, 17, y)
    cell.font, cell.border, cell.alignment = BODY, BR, CTR
print(f"Scoring column Q: {len(ROWS)} static Yes-counts written.")

# ---- job 2: rebuild Summary with static values ---------------------------
idx = wb.sheetnames.index("Summary") if "Summary" in wb.sheetnames else 3
if "Summary" in wb.sheetnames:
    del wb["Summary"]
ws = wb.create_sheet("Summary", idx)
ws.sheet_properties.tabColor = "1F3864"


def put(r, c, v, font=BODY, fmt=None, align=None, fill=None):
    cell = ws.cell(r, c, v)
    cell.font, cell.border = font, BR
    cell.alignment = align or WRAP
    if fmt:
        cell.number_format = fmt
    if fill:
        cell.fill = fill


def section(r, text):
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=9)
    c = ws.cell(r, 1, text)
    c.font, c.fill, c.border = SFONT, SFILL, BR
    for cc in range(2, 10):
        ws.cell(r, cc).fill, ws.cell(r, cc).border = SFILL, BR


ws.merge_cells("A1:I1")
t = ws.cell(1, 1, "SUMMARY \u2014 Rigor appraisal of AI-based imaging prediction tools")
t.font = Font(name=F, bold=True, size=13, color="FFFFFF")
t.fill = HFILL
t.alignment = Alignment(vertical="center", horizontal="left")
ws.row_dimensions[1].height = 26

ws.merge_cells("A2:I2")
s = ws.cell(2, 1,
            f"n = 15 scored papers (Retinal 4, Colonoscopy 7, ECG 4)   |   "
            f"Percentages = Yes / (Yes + No + Unclear); N/A and unscored cells excluded from both "
            f"numerator and denominator   |   STATIC VALUES, last refreshed {STAMP} \u2014 "
            f"re-run Notebooks/refresh_tracker.ipynb after any change to the Scoring tab")
s.font = ITAL
s.alignment = Alignment(wrap_text=True, vertical="center")
ws.row_dimensions[2].height = 30

# --- A. per item ---------------------------------------------------------
r = 4
section(r, "A.  PROPORTION MEETING EACH CHECKLIST ITEM")
r += 1
for c, h in enumerate(["#", "Checklist item", "Overall", "Retinal", "Colonoscopy", "ECG",
                       "Yes", "No", "Unclear"], start=1):
    put(r, c, h, HFONT, align=CTR, fill=HFILL)
a0 = r + 1
for i, item in enumerate(ITEMS):
    r += 1
    col = 4 + i
    y, d, pc = rate(ROWS, col)
    t3 = Counter(sc.cell(rr, col).value for rr in ROWS)
    put(r, 1, i + 1, BODY, align=CTR)
    put(r, 2, item)
    put(r, 3, pc, BOLD, "0%", CTR)
    for j, dom in enumerate(("Retinal", "Colonoscopy", "ECG")):
        put(r, 4 + j, rate(DOMS[dom], col)[2], BODY, "0%", CTR)
    for k, v in enumerate(("Yes", "No", "Unclear")):
        put(r, 7 + k, t3[v], BODY, "0", CTR)
ws.conditional_formatting.add(f"C{a0}:F{r}", ColorScaleRule(**SCALE))

# --- B. item groups ------------------------------------------------------
r += 2
section(r, "B.  ITEM GROUPS \u2014 THE HYPOTHESIS TEST")
r += 1
for c, h in enumerate(["", "Group", "All 15", "PRIMARY (imaging, 11)",
                       "SECONDARY (ECG, 4)", "", "Yes", "No", "Unclear"], start=1):
    put(r, c, h, HFONT, align=CTR, fill=HFILL)
b0 = r + 1
group_rows = []
for name, idxs in GROUPS:
    r += 1
    ty = td = tn = tu = 0
    for i in idxs:
        t3 = Counter(sc.cell(rr, 4 + i).value for rr in ROWS)
        ty += t3["Yes"]
        tn += t3["No"]
        tu += t3["Unclear"]
        td += t3["Yes"] + t3["No"] + t3["Unclear"]
    def set_rate(rows_subset):
        sy = sd = 0
        for i in idxs:
            t = Counter(sc.cell(rr, 4 + i).value for rr in rows_subset)
            sy += t["Yes"]
            sd += t["Yes"] + t["No"] + t["Unclear"]
        return (sy / sd if sd else None), sy, sd

    p_rate, p_y, p_d = set_rate(PRIMARY_ROWS)
    s_rate, s_y, s_d = set_rate(SECONDARY_ROWS)
    group_rows.append((name, ty, td, p_rate, p_y, p_d, s_rate, s_y, s_d))
    put(r, 1, "")
    put(r, 2, name)
    put(r, 3, ty / td if td else None, BODY, "0%", CTR)
    put(r, 4, p_rate, BOLD, "0%", CTR)
    put(r, 5, s_rate, BODY, "0%", CTR)
    put(r, 6, "")
    put(r, 7, ty, BODY, "0", CTR)
    put(r, 8, tn, BODY, "0", CTR)
    put(r, 9, tu, BODY, "0", CTR)
ws.conditional_formatting.add(f"C{b0}:E{r}", ColorScaleRule(**SCALE))

# --- C. per paper --------------------------------------------------------
r += 2
section(r, "C.  PER-PAPER SCORES")
r += 1
for c, h in enumerate(["#", "Paper", "Rate", "Domain", "", "", "Yes", "Scoreable", ""], start=1):
    put(r, c, h, HFONT, align=CTR, fill=HFILL)
c0 = r + 1
for sr in ROWS:
    r += 1
    v = [sc.cell(sr, c).value for c in range(4, 17)]
    y = sum(1 for x in v if x == "Yes")
    d = sum(1 for x in v if x in SCOREABLE)
    put(r, 1, sc.cell(sr, 1).value, BODY, "0", CTR)
    put(r, 2, sc.cell(sr, 3).value)
    put(r, 3, y / d if d else None, BOLD, "0%", CTR)
    put(r, 4, sc.cell(sr, 2).value, BODY, align=CTR)
    for c in (5, 6):
        put(r, c, "")
    put(r, 7, y, BODY, "0", CTR)
    put(r, 8, d, BODY, "0", CTR)
    put(r, 9, "")
ws.conditional_formatting.add(f"C{c0}:C{r}", ColorScaleRule(**SCALE))

# --- D. diagnostics ------------------------------------------------------
r += 2
section(r, "D.  COLUMN DIAGNOSTICS \u2014 can each item discriminate?")
r += 1
for c, h in enumerate(["#", "Checklist item", "Distinct verdicts used", "Status",
                       "", "", "", "", ""], start=1):
    put(r, c, h, HFONT, align=CTR, fill=HFILL)
for i, item in enumerate(ITEMS):
    r += 1
    col = 4 + i
    t3 = Counter(sc.cell(rr, col).value for rr in ROWS)
    distinct = sum(1 for k in SCOREABLE if t3[k])
    _, _, pc = rate(ROWS, col)
    status = ("DEGENERATE \u2014 cannot discriminate" if distinct == 1
              else "Near-degenerate" if (pc is not None and (pc >= 0.9 or pc <= 0.1))
              else "Discriminates")
    put(r, 1, i + 1, BODY, align=CTR)
    put(r, 2, item)
    put(r, 3, distinct, BODY, "0", CTR)
    put(r, 4, status)
    for c in range(5, 10):
        put(r, c, "")

# --- notes ---------------------------------------------------------------
NOTES = [
    "These figures are STATIC VALUES written by Notebooks/refresh_tracker.ipynb, not live formulas. "
    "Live formulas were tried twice and rendered as 0% in Excel, because openpyxl strips the cached "
    "result and Excel would not honour the recalculate-on-load flag. Re-run the notebook after any "
    "change to the Scoring tab.",
    "Scoring conventions are recorded in the Search Protocol sheet, section I, and in the Legend. "
    "Item classes: A (always Yes/No) = 2, 9; B (N/A absent trial arms) = 3, 4, 5, 11, 12; "
    "C (N/A absent any impact evaluation) = 7; D (scored for every paper) = 1, 6, 8, 10, 13.",
    "Unregistered trials score Unclear on items 3 and 4, because self-reported prespecification "
    "cannot be verified. Trials registered prospectively outside ClinicalTrials.gov score Unclear "
    "on item 5 (Alali, Kuwaiti registry; Mathenge, PACTR).",
    "Five papers are excluded from every figure and held as supporting evidence in Papers/Supporting/. "
    "Two were manifest errors caught by page-one verification, two failed screening criteria, and one "
    "was eligible but over quota. See the Screening Log.",
    "INSTRUMENT LIMITATION 1 \u2014 item 13 (No Reverse Causation) records a single verdict across all 15 "
    "papers and cannot discriminate. Authors do not report their own data leakage, so the item is "
    "unanswerable from a published paper.",
    "INSTRUMENT LIMITATION 2 \u2014 item 8 (No Conflict of Interest) returned zero No verdicts across 15 "
    "papers, including six with manufacturer employees, directors, shareholders or licensing interests "
    "among the authors. The item detects DISCLOSURE, not independence. Both non-Yes verdicts (Tsai, Liu) "
    "were reachable only by cross-referencing a companion paper, which single-paper appraisal cannot do.",
    "INSTRUMENT LIMITATION 3 \u2014 item 2 (Patient-Level RCT) scores No for cluster-randomised designs "
    "(Yao, Liu) which are arguably MORE rigorous for clinician-facing tools because they eliminate "
    "contamination. Relatedly, Ladabaum 2023 scores lowest in the set while being its most candid paper, "
    "because a deliberately minimal real-world design forfeits five items at once.",
    "EMPIRICAL PATTERN \u2014 7 of the 12 randomised trials built their power calculation on a baseline "
    "event rate the observed data contradicted. Only Liu 2025 met its enrolment target outright, and "
    "only RAIDERS came within one participant of doing so.",
]
r += 2
section(r, "NOTES")
for note in NOTES:
    r += 1
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=9)
    c = ws.cell(r, 1, note)
    c.font, c.alignment, c.border = ITAL, WRAP, BR
    for cc in range(2, 10):
        ws.cell(r, cc).border = BR
    ws.row_dimensions[r].height = 30

for col, w in zip("ABCDEFGHI", (5, 58, 11, 12, 13, 11, 8, 10, 9)):
    ws.column_dimensions[col].width = w
ws.freeze_panes = "A3"

wb.save(XL)
print(f"Summary rebuilt with static values ({r} rows). Stamp: {STAMP}")

Scoring column Q: 15 static Yes-counts written.


Summary rebuilt with static values (71 rows). Stamp: 2026-08-08 18:05


## Regenerate the CSV exports

Four files under `Data/Tracker/exports/`, computed straight from the Scoring grid so they never depend on Excel evaluating anything.

`scoring_long.csv` is the tidy format, one row per paper-item pair with `is_yes` and `scoreable` as 0/1 indicators. That is the one to load for proportions, confidence intervals or a chi-square.

In [4]:
OUT.mkdir(parents=True, exist_ok=True)

with open(OUT / "scoring_wide.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["#", "Domain", "set", "Citation"] + ITEMS + ["Yes", "Scoreable", "Rate"])
    for rr in ROWS:
        v = [sc.cell(rr, c).value for c in range(4, 17)]
        y = sum(1 for x in v if x == "Yes")
        d = sum(1 for x in v if x in SCOREABLE)
        w.writerow([sc.cell(rr, 1).value, sc.cell(rr, 2).value,
                    set_of(sc.cell(rr, 2).value), sc.cell(rr, 3).value]
                   + v + [y, d, round(y / d, 4) if d else ""])

with open(OUT / "scoring_long.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["paper_id", "domain", "set", "citation", "item_no", "item",
                "verdict", "scoreable", "is_yes"])
    for rr in ROWS:
        for i, item in enumerate(ITEMS):
            v = sc.cell(rr, 4 + i).value
            w.writerow([sc.cell(rr, 1).value, sc.cell(rr, 2).value,
                        set_of(sc.cell(rr, 2).value), sc.cell(rr, 3).value,
                        i + 1, item, v, int(v in SCOREABLE), int(v == "Yes")])

with open(OUT / "summary_by_item.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["item_no", "item", "overall_yes", "overall_scoreable", "overall_rate",
                "retinal_rate", "colonoscopy_rate", "ecg_rate",
                "n_yes", "n_no", "n_unclear", "n_na", "distinct_verdicts", "status"])
    for i, item in enumerate(ITEMS):
        col = 4 + i
        y, d, pc = rate(ROWS, col)
        t3 = Counter(sc.cell(rr, col).value for rr in ROWS)
        distinct = sum(1 for k in SCOREABLE if t3[k])
        status = ("DEGENERATE" if distinct == 1
                  else "Near-degenerate" if (pc is not None and (pc >= 0.9 or pc <= 0.1))
                  else "Discriminates")
        dom_rates = [rate(DOMS[d2], col)[2] for d2 in ("Retinal", "Colonoscopy", "ECG")]
        w.writerow([i + 1, item, y, d, round(pc, 4) if pc is not None else "",
                    *[round(x, 4) if x is not None else "" for x in dom_rates],
                    t3["Yes"], t3["No"], t3["Unclear"], t3["N/A"], distinct, status])

with open(OUT / "summary_by_group.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["group", "items", "yes", "scoreable", "rate",
                "primary_yes", "primary_scoreable", "primary_rate",
                "secondary_yes", "secondary_scoreable", "secondary_rate"])
    for (name, idxs), (_, ty, td, p_rate, p_y, p_d, s_rate, s_y, s_d) in zip(GROUPS, group_rows):
        w.writerow([name.split("\u2014")[0].strip().rstrip(","),
                    " ".join(str(i + 1) for i in idxs), ty, td, round(ty / td, 4),
                    p_y, p_d, round(p_rate, 4) if p_rate is not None else "",
                    s_y, s_d, round(s_rate, 4) if s_rate is not None else ""])

print(f"Exports refreshed in {OUT}:")
for fn in sorted(OUT.iterdir()):
    print(f"  {fn.name:<24} {fn.stat().st_size:>7,} bytes")

Exports refreshed in C:\Users\adamt\Desktop\Capstone - AI Imaging Appraisal\Data\Tracker\exports:
  agreement_summary.csv        466 bytes
  ci_by_domain.csv           2,662 bytes
  ci_by_domain_secondary.csv   1,172 bytes
  ci_by_group.csv            1,468 bytes
  ci_by_group_secondary.csv   1,304 bytes
  ci_by_item.csv             1,806 bytes
  ci_by_item_secondary.csv   1,536 bytes
  ci_contrasts.csv             642 bytes
  ci_contrasts_secondary.csv     487 bytes
  disagreements.csv          6,663 bytes
  scoring_long.csv          20,917 bytes
  scoring_wide.csv           2,427 bytes
  self_consistency.csv       2,323 bytes
  summary_by_group.csv         476 bytes
  summary_by_item.csv        1,142 bytes


In [5]:
print("Values now written into the Summary tab:\n")
for i, item in enumerate(ITEMS):
    y, d, pc = rate(ROWS, 4 + i)
    print(f"  {i+1:>2}. {item:<26} {pc:>6.0%}  ({y}/{d})")

print()
for name, ty, td, p_rate, p_y, p_d, s_rate, s_y, s_d in group_rows:
    label = name.split("\u2014")[0].strip()
    label = name.split("\u2014")[0].strip().rstrip(",")
    pr = f"{p_rate*100:3.0f}%" if p_rate is not None else "  -"
    sr = f"{s_rate*100:3.0f}%" if s_rate is not None else "  -"
    print(f"  {label:<34}{ty/td*100:4.0f}%  all15   "
          f"{pr} primary ({p_y}/{p_d})   {sr} secondary ({s_y}/{s_d})")

print("\nOpen the workbook and confirm the Summary tab matches the figures above.")

Values now written into the Summary tab:

   1. External Validation           93%  (14/15)
   2. Patient-Level RCT             67%  (10/15)
   3. Prespecified Endpoint         79%  (11/14)
   4. Prespecified Design           71%  (10/14)
   5. Registered w/ CT.gov          71%  (10/14)
   6. Transparent Reporting          0%  (0/15)
   7. Honest Impact Eval            85%  (11/13)
   8. No Conflict of Interest       27%  (4/15)
   9. Focus on Patient Outcomes      7%  (1/15)
  10. Adequate Sample Size          71%  (10/14)
  11. Intention-to-Treat            85%  (11/13)
  12. Balanced Table 1              85%  (11/13)
  13. No Reverse Causation         100%  (14/14)

  MODEL-DEVELOPMENT                   97%  all15    95% primary (20/21)   100% secondary (8/8)
  TRIAL CONDUCT                       77%  all15    72% primary (42/58)    88% secondary (21/24)
  INTEGRITY                           54%  all15    55% primary (11/20)    50% secondary (4/8)
  CLINICAL IMPACT                   

## Operating notes

- **Run this after any change to the Scoring tab**, whether made by hand or by a script. Nothing recalculates on its own.
- **The last-refreshed stamp** is in the Summary subtitle row. If it looks older than your most recent edit, the figures are stale.
- **The Scoring tab keeps its dropdown validation** on D2:P16. Those are data validations, not formulas, so they survive every rebuild.
- **This notebook never reads PDFs and never assigns verdicts.** That is `score_papers.ipynb`, which is entirely separate and shares no code.
- **If Excel has the workbook open**, the first cell stops with a clear message rather than failing halfway through a rebuild.